# Run all notebooks in the project

In [ ]:
from pathlib import Path
import csv

import cv2
import numpy as np
import matplotlib.pyplot as plt
from mediapipe import Image, ImageFormat
from mediapipe.tasks import python, vision

# Files are located beside this script
folder = Path(__file__).resolve().parent
video_path = folder / "squat.mp4"
model_path = folder / "pose_landmarker_lite.task"

# Change to "right" if the right side faces the camera
SIDE = "left"

# Shoulder, hip, knee, ankle, foot index
indices = {
    "left": (11, 23, 25, 27, 31),
    "right": (12, 24, 26, 28, 32),
}[SIDE]


def angle(a, b, c):
    """2D included angle at point b, in degrees."""
    u = np.asarray(a) - np.asarray(b)
    v = np.asarray(c) - np.asarray(b)
    length = np.linalg.norm(u) * np.linalg.norm(v)
    if length < 1e-6:
        return np.nan
    cosine = np.clip(np.dot(u, v) / length, -1, 1)
    return float(np.degrees(np.arccos(cosine)))


options = vision.PoseLandmarkerOptions(
    base_options=python.BaseOptions(model_asset_path=str(model_path)),
    running_mode=vision.RunningMode.VIDEO,
    num_poses=1,
)

cap = cv2.VideoCapture(str(video_path))
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {video_path}")

fps = cap.get(cv2.CAP_PROP_FPS)
if not np.isfinite(fps) or fps <= 0:
    cap.release()
    raise RuntimeError("Cannot read video frame rate")

writer = None
rows = []
frame_number = 0
last_timestamp = -1

try:
    with vision.PoseLandmarker.create_from_options(options) as detector:
        while True:
            success, frame = cap.read()
            if not success:
                break

            height, width = frame.shape[:2]

            if writer is None:
                writer = cv2.VideoWriter(
                    str(folder / "result.mp4"),
                    cv2.VideoWriter_fourcc(*"mp4v"),
                    fps, (width, height),
                )
                if not writer.isOpened():
                    raise RuntimeError("Cannot create output video")

            seconds = frame_number / fps
            timestamp = max(last_timestamp + 1, round(seconds * 1000))
            last_timestamp = timestamp

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image = Image(image_format=ImageFormat.SRGB, data=rgb)
            result = detector.detect_for_video(image, timestamp)

            values = [np.nan, np.nan, np.nan]
            message = "Joints not clearly visible"

            if result.pose_landmarks:
                landmarks = result.pose_landmarks[0]
                joints = [landmarks[i] for i in indices]

                # Skip unreliable measurements
                reliable = all(
                    (p.visibility or 0) >= 0.5
                    and (p.presence or 0) >= 0.5
                    for p in joints
                )

                if reliable:
                    points = [
                        (p.x * width, p.y * height) for p in joints
                    ]
                    shoulder, hip, knee, ankle, foot = points

                    values = [
                        angle(shoulder, hip, knee),
                        angle(hip, knee, ankle),
                        angle(knee, ankle, foot),
                    ]

                    pixels = [
                        tuple(round(v) for v in point)
                        for point in points
                    ]
                    for a, b in zip(pixels[:-1], pixels[1:]):
                        cv2.line(frame, a, b, (0, 255, 0), 3)
                    for point in pixels:
                        cv2.circle(frame, point, 6, (0, 255, 255), -1)

                    message = (
                        f"Hip: {values[0]:.1f}  "
                        f"Knee: {values[1]:.1f}  "
                        f"Ankle: {values[2]:.1f}"
                    )

            cv2.putText(
                frame, message, (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.65, (0, 0, 255), 2,
            )
            writer.write(frame)
            rows.append([seconds, *values])
            frame_number += 1

            if frame_number % 30 == 0:
                print(f"Processed {seconds:.1f} seconds")

finally:
    cap.release()
    if writer is not None:
        writer.release()

if not rows:
    raise RuntimeError("No frames found in video")

with open(folder / "angles.csv", "w", newline="") as file:
    csv_writer = csv.writer(file)
    csv_writer.writerow([
        "time_s", "hip_angle_deg", "knee_angle_deg", "ankle_angle_deg"
    ])
    csv_writer.writerows(rows)

data = np.asarray(rows)
for column, label in enumerate(["Hip", "Knee", "Ankle"], start=1):
    plt.plot(data[:, 0], data[:, column], label=label)

plt.xlabel("Time (s)")
plt.ylabel("2D included angle (degrees)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(folder / "angles.png", dpi=150)
plt.close()

print("Finished: result.mp4, angles.csv, angles.png")